# 📊 Importar KM Dinâmico - Multi Anos (2023-2026)

Script que:
- ✅ Processa anos 2023, 2024, 2025, 2026
- ✅ Cria tabela separada por ano
- ✅ Remove duplicados do dataframe
- ✅ Verifica registos já existentes
- ✅ Insere APENAS registos novos
- ✅ Usa UNIQUE constraint para evitar duplicados

In [1]:
import os
import pandas as pd
import openpyxl
from datetime import datetime
import sqlite3
import platform

print("✅ Imports carregados")

✅ Imports carregados


In [2]:
# ── CONFIGURAÇÕES ─────────────────────────────────────────────────────────
ANOS = [2023, 2024, 2025, 2026]
BASE_PATH_ROOT = r"T:\Portugal\D-Trafico\KM BASE"

MONTH_FILES = {
    1:  "01 JANEIRO.xlsx",
    2:  "02 FEVEREIRO.xlsx",
    3:  "03 MARÇO.xlsx",
    4:  "04 ABRIL.xlsx",
    5:  "05 MAIO.xlsx",
    6:  "06 JUNHO.xlsx",
    7:  "07 JULHO.xlsx",
    8:  "08 AGOSTO.xlsx",
    9:  "09 Setembro.xlsx",
    10: "10 Outubro.xlsx",
    11: "11 Novembro.xlsx",
    12: "12 Dezembro.xlsx",
}

if platform.system() == 'Windows':
    DB_PATH = r"C:\Users\LISARR\Documents\python\01.Financeiro\inform_27.db"
elif platform.system() == 'Darwin':
    DB_PATH = "/Volumes/RR/DB/inform_27.db"
else:
    DB_PATH = "inform_27.db"

conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()

print(f"📂 Base path: {BASE_PATH_ROOT}")
print(f"📅 Anos a processar: {ANOS}")
print(f"📋 Meses por ano: {len(MONTH_FILES)}")
print(f"💾 BD: {DB_PATH}")

📂 Base path: T:\Portugal\D-Trafico\KM BASE
📅 Anos a processar: [2023, 2024, 2025, 2026]
📋 Meses por ano: 12
💾 BD: C:\Users\LISARR\Documents\python\01.Financeiro\inform_27.db


In [3]:
# ── FUNÇÕES DE PARSING ────────────────────────────────────────────────────
def clean_value(val):
    """Limpar e converter valores."""
    if val is None or (isinstance(val, float) and pd.isna(val)):
        return 0
    if isinstance(val, (int, float)):
        return float(val)
    return 0

def parse_formula(formula_str):
    """Extrair tipo_contrato, km_gratis e rate_km da fórmula."""
    if not formula_str:
        return None, 0, 0
    
    tipo_formula = "desconhecido"
    km_gratis = 0
    rate = 0
    
    if "fixo" in str(formula_str).lower():
        tipo_formula = "fixo"
    elif "km" in str(formula_str).lower():
        tipo_formula = "km"
    
    if "-" in str(formula_str):
        try:
            parts = str(formula_str).split("-")
            if len(parts) >= 2:
                km_gratis = float(parts[1].replace(")", "").strip())
        except:
            pass
    
    return tipo_formula, km_gratis, rate

def parse_cell_a(cell_value, sheet_name, row_idx, anon_map):
    """Extrair transportador e viatura da célula A."""
    if not cell_value:
        return None, None, None
    
    cell_str = str(cell_value).strip()
    parts = cell_str.split("-", 1)
    
    transportador = parts[0].strip() if len(parts) > 0 else sheet_name
    viatura_raw = parts[1].strip() if len(parts) > 1 else None
    
    if not viatura_raw:
        return None, None, None
    
    viatura = viatura_raw
    tipo_veiculo = "DESCONHECIDO"
    
    return transportador, viatura, tipo_veiculo

def extract_sheet_data(ws_values, ws_formulas, sheet_name, month):
    """Extrair dados da folha de cálculo."""
    sheet_data = []
    
    first_row = list(ws_values.iter_rows(min_row=1, max_row=1, values_only=True))[0]
    date_columns = {}
    
    for col_idx in range(2, len(first_row)):
        val = first_row[col_idx]
        if isinstance(val, datetime):
            date_columns[col_idx] = val
    
    if not date_columns:
        return sheet_data
    
    anon_map = {}
    row_idx = 2
    
    while row_idx <= ws_values.max_row:
        current_row = list(ws_values.iter_rows(
            min_row=row_idx, max_row=row_idx, values_only=True))[0]
        
        if not current_row or not current_row[0]:
            row_idx += 1
            continue
        
        col_b = str(current_row[1]).strip().upper() if current_row[1] else ""
        if col_b != "KM":
            row_idx += 1
            continue
        
        transportador, viatura, tipo_veiculo = parse_cell_a(
            current_row[0], sheet_name, row_idx, anon_map)
        
        if not viatura:
            row_idx += 1
            continue
        
        km_row = current_row
        p_row = list(ws_values.iter_rows(min_row=row_idx+1, max_row=row_idx+1, values_only=True))[0]
        vt_row = list(ws_values.iter_rows(min_row=row_idx+2, max_row=row_idx+2, values_only=True))[0]
        sd_row = list(ws_values.iter_rows(min_row=row_idx+3, max_row=row_idx+3, values_only=True))[0]
        tot_row = list(ws_values.iter_rows(min_row=row_idx+4, max_row=row_idx+4, values_only=True))[0]
        
        formula_cell = ws_formulas.cell(row=row_idx + 3, column=3).value
        tipo_formula, km_gratis, rate = parse_formula(formula_cell)
        
        for col_idx, date_obj in date_columns.items():
            if col_idx >= len(km_row):
                continue
            
            km_value = clean_value(km_row[col_idx])
            if not km_value or km_value == 0:
                continue
            
            portagens = clean_value(p_row[col_idx] if col_idx < len(p_row) else None)
            valor_total = clean_value(vt_row[col_idx] if col_idx < len(vt_row) else None)
            sub_divisao = clean_value(sd_row[col_idx] if col_idx < len(sd_row) else None)
            preco_base = valor_total - sub_divisao if (valor_total and sub_divisao) else valor_total
            
            sheet_data.append({
                'transportador': transportador,
                'tipo_veiculo': tipo_veiculo,
                'viatura': viatura,
                'data': date_obj,
                'dia': date_obj.day,
                'mes': month,
                'km': int(km_value) if km_value else 0,
                'portagens': portagens,
                'preco_base': preco_base,
                'sub_divisao': sub_divisao,
                'tipo_contrato': tipo_formula,
                'km_gratis': km_gratis,
                'rate_km': rate,
                'total': valor_total,
            })
        
        row_idx += 5
    
    return sheet_data

print("✅ Funções carregadas")

✅ Funções carregadas


In [4]:
# ── PROCESSAR CADA ANO ────────────────────────────────────────────────────
for ano in ANOS:
    print(f"\n{'='*70}")
    print(f"📅 PROCESSANDO ANO {ano}")
    print(f"{'='*70}\n")
    
    BASE_PATH = rf"{BASE_PATH_ROOT}\Ano {ano}"
    table_name = f"km_diario_{ano}"
    
    if not os.path.exists(BASE_PATH):
        print(f"   ⚠️  Pasta não encontrada: {BASE_PATH}\n")
        continue
    
    # Criar tabela
    print(f"🔨 Criando tabela '{table_name}'...\n")
    cursor.execute(f"""
        CREATE TABLE IF NOT EXISTS {table_name} (
            id              INTEGER PRIMARY KEY AUTOINCREMENT,
            transportador   TEXT NOT NULL,
            tipo_veiculo    TEXT,
            viatura         TEXT NOT NULL,
            data            DATE NOT NULL,
            dia             INTEGER,
            mes             INTEGER,
            km              INTEGER,
            portagens       REAL,
            preco_base      REAL,
            sub_divisao     REAL,
            tipo_contrato   TEXT,
            km_gratis       REAL,
            rate_km         REAL,
            total           REAL,
            criado_em       TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
            UNIQUE(transportador, viatura, data)
        )
    """)
    conn.commit()
    
    # Ler ficheiros do ano
    all_data = []
    
    for month, filename in MONTH_FILES.items():
        filepath = os.path.join(BASE_PATH, filename)
        
        if not os.path.exists(filepath):
            continue
        
        try:
            wb_val = openpyxl.load_workbook(filepath, data_only=True)
            wb_form = openpyxl.load_workbook(filepath, data_only=False)
            
            data = []
            for sheet_name in wb_val.sheetnames:
                ws_v = wb_val[sheet_name]
                ws_f = wb_form[sheet_name]
                sheet_data = extract_sheet_data(ws_v, ws_f, sheet_name.strip(), month)
                data.extend(sheet_data)
            
            all_data.extend(data)
            wb_val.close()
            wb_form.close()
            print(f"   ✅ {filename}: {len(data)} registos")
        except Exception as e:
            print(f"   ❌ {filename}: {e}")
    
    if not all_data:
        print(f"   ⚠️  Nenhum dado encontrado para {ano}\n")
        continue
    
    df = pd.DataFrame(all_data)
    
    # Remover duplicados do dataframe
    print(f"\n🔍 Removendo duplicados do dataframe...")
    df_before = len(df)
    df = df.drop_duplicates(
        subset=['transportador', 'viatura', 'data'], 
        keep='first'
    )
    df_removed = df_before - len(df)
    
    if df_removed > 0:
        print(f"   ⚠️  {df_removed} registos removidos")
    else:
        print(f"   ✅ Nenhum duplicado")
    
    # Verificar registos já existentes
    print(f"\n🔍 Verificando registos já existentes...")
    existing = cursor.execute(f"""
        SELECT transportador, viatura, DATE(data) FROM {table_name}
    """).fetchall()
    
    existing_set = {(t[0], t[1], t[2]) for t in existing}
    print(f"   📊 Já existem: {len(existing_set):,}")
    
    # Filtrar novos registos
    df['data_str'] = df['data'].dt.strftime('%Y-%m-%d')
    df['_key'] = df.apply(
        lambda x: (x['transportador'], x['viatura'], x['data_str']), 
        axis=1
    )
    
    df_insert = df[~df['_key'].isin(existing_set)].copy()
    df_insert = df_insert.drop(['data_str', '_key'], axis=1)
    
    print(f"   ➡️  Novos registos: {len(df_insert):,}")
    print(f"   ⏭️  Já na BD: {len(df) - len(df_insert):,}")
    
    # Inserir
    if len(df_insert) > 0:
        try:
            df_insert.to_sql(
                name=table_name,
                con=conn,
                if_exists='append',
                index=False,
            )
            conn.commit()
            print(f"   ✅ Inserção concluída")
        except Exception as e:
            print(f"   ❌ Erro: {e}")
            conn.rollback()
    else:
        print(f"   ⏭️  Nenhum novo registro para inserir")
    
    # Resumo
    total = cursor.execute(f"SELECT COUNT(*) FROM {table_name}").fetchone()[0]
    print(f"\n   📊 Total na tabela '{table_name}': {total:,}\n")

conn.close()
print(f"{'='*70}")
print(f"✅ Processo concluído com sucesso!")
print(f"{'='*70}")


📅 PROCESSANDO ANO 2023

🔨 Criando tabela 'km_diario_2023'...

   ✅ 01 JANEIRO.xlsx: 233 registos
   ✅ 03 MARÇO.xlsx: 415 registos
   ✅ 04 ABRIL.xlsx: 361 registos
   ✅ 05 MAIO.xlsx: 407 registos
   ✅ 06 JUNHO.xlsx: 432 registos
   ✅ 07 JULHO.xlsx: 458 registos
   ✅ 08 AGOSTO.xlsx: 485 registos
   ✅ 09 Setembro.xlsx: 445 registos
   ✅ 10 Outubro.xlsx: 444 registos
   ✅ 11 Novembro.xlsx: 445 registos
   ✅ 12 Dezembro.xlsx: 409 registos

🔍 Removendo duplicados do dataframe...
   ✅ Nenhum duplicado

🔍 Verificando registos já existentes...
   📊 Já existem: 4,534
   ➡️  Novos registos: 0
   ⏭️  Já na BD: 4,534
   ⏭️  Nenhum novo registro para inserir

   📊 Total na tabela 'km_diario_2023': 4,534


📅 PROCESSANDO ANO 2024

🔨 Criando tabela 'km_diario_2024'...

   ✅ 01 JANEIRO.xlsx: 424 registos
   ✅ 02 FEVEREIRO.xlsx: 411 registos
   ✅ 03 MARÇO.xlsx: 432 registos
   ✅ 04 ABRIL.xlsx: 453 registos
   ✅ 05 MAIO.xlsx: 468 registos
   ✅ 06 JUNHO.xlsx: 428 registos
   ✅ 07 JULHO.xlsx: 471 registos
